In [3]:
from transformers import AutoTokenizer, AutoModel

In [4]:
colbert_tokenizer = AutoTokenizer.from_pretrained("colbert-ir/colbertv2.0")
colbert = AutoModel.from_pretrained("colbert-ir/colbertv2.0")

In [5]:
query = "What are the health benefits of meditation?"

documents = [
    "Several clinical studies have shown that regular meditation can help reduce stress and anxiety levels.",
    "Meditation involves focusing the mind and eliminating distractions, often through breathing techniques or guided imagery.",
    "A daily meditation practice has been associated with lower blood pressure and improved sleep quality in adults.",
    "The city of Kyoto is famous for its Zen temples, where meditation has been practiced for centuries.",
    "People who meditate frequently often report feeling calmer and more focused throughout the day.",
    "Research suggests meditation may lower the risk of heart disease by reducing inflammation and improving heart rate variability.",
    "Meditation apps have become increasingly popular, offering guided sessions on mindfulness and relaxation.",
    "A 2021 meta-analysis found that meditation can reduce symptoms of depression when used alongside other treatments.",
    "Some forms of meditation emphasize compassion and kindness, aiming to improve emotional well-being.",
    "Athletes sometimes use meditation techniques to enhance concentration and mental resilience during competition.",
]

In [6]:
import torch
import torch.nn.functional as F

def get_token_embeddings(text, tokenizer, model):
    # Get token-level embeddings, ignore [CLS] and [SEP]
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # outputs.last_hidden_state: (1, seq_len, hidden_dim)
    input_ids = inputs['input_ids'][0]
    keep_indices = (input_ids != tokenizer.cls_token_id) & (input_ids != tokenizer.sep_token_id)
    
    return outputs.last_hidden_state[0][keep_indices]  # (filtered_seq_len, hidden_dim)

def colbert_score(query_emb, doc_emb):
    
    # query_emb: (m, d), doc_emb: (n, d)
    sim = F.cosine_similarity(query_emb.unsqueeze(1), doc_emb.unsqueeze(0), dim=2)  # (m, n)
    
    max_sim, _ = sim.max(dim=1)  # (m,)
    
    return max_sim.sum().item()

In [7]:
# Compute token level embeddings
query_emb = get_token_embeddings(query, colbert_tokenizer, colbert)
doc_embs = [get_token_embeddings(doc, colbert_tokenizer, colbert) for doc in documents]

In [10]:
print(query_emb.shape)
print(query_emb.unsqueeze(1).shape)

torch.Size([8, 768])
torch.Size([8, 1, 768])


In [11]:
print(doc_embs[0].shape)
print(doc_embs[0].unsqueeze(0).shape)

torch.Size([16, 768])
torch.Size([1, 16, 768])


In [12]:
# Compute scores
scores = [colbert_score(query_emb, doc_emb) for doc_emb in doc_embs]

ranking = sorted(zip(scores, documents), reverse=True)

print("="*25, "Late Interaction Rankings", "="*25, "\n")
for score, doc in ranking:
    print(f"{score:.2f}\t{doc}")

========================= Late Interaction Rankings ========================= 

6.02	Research suggests meditation may lower the risk of heart disease by reducing inflammation and improving heart rate variability.
5.93	A daily meditation practice has been associated with lower blood pressure and improved sleep quality in adults.
5.88	Several clinical studies have shown that regular meditation can help reduce stress and anxiety levels.
5.71	A 2021 meta-analysis found that meditation can reduce symptoms of depression when used alongside other treatments.
5.63	Some forms of meditation emphasize compassion and kindness, aiming to improve emotional well-being.
5.12	Meditation apps have become increasingly popular, offering guided sessions on mindfulness and relaxation.
5.10	Athletes sometimes use meditation techniques to enhance concentration and mental resilience during competition.
4.91	Meditation involves focusing the mind and eliminating distractions, often through breathing techniques o